# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akhilabusalih/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

This playbook ranks existing content for human review. The queue score combines the validated model probability with the transparent baseline score; it is a prioritization score, not a probability that a refresh will succeed.

The reason codes make each recommendation inspectable:

- `model_decline_risk`: the model flags elevated observed decline risk.
- `declining_with_demand`: the observed trend label is down while the page has demand.
- `stale_visible_page`: an older page still has measurable visibility.
- `ctr_review_candidate`: visible impressions and position coexist with low CTR.
- `engagement_review_candidate`: measurable sessions coexist with low engagement or scroll rate.
- `thin_visible_page`: a visible page may warrant content expansion review.

The action mapping stays deliberately narrow: refresh, refresh with CTR review, refresh with engagement review, expand and refresh, or monitor. A reason code explains why an item entered the queue; it does not prescribe an automatic edit.

In [1]:
from __future__ import annotations

import json
import shutil
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "scripts").exists() and (candidate / "data").exists():
            return candidate
    raise RuntimeError("Could not locate repository root")


ROOT = find_repo_root(Path.cwd())
RAW_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
PIPELINE_OUTPUT = ROOT / "outputs"
WORK_OUTPUT = ROOT / "work" / "outputs"
WORK_FIGURES = ROOT / "work" / "figures"
WORK_OUTPUT.mkdir(parents=True, exist_ok=True)
WORK_FIGURES.mkdir(parents=True, exist_ok=True)
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))

# Regenerate ignored artifacts from the committed anonymized starter dataset.
for script_name in [
    "01_prepare_features.py",
    "02_baseline_score.py",
    "03_train_model.py",
    "04_evaluate_and_export.py",
]:
    subprocess.run([sys.executable, str(ROOT / "scripts" / script_name)], check=True)

queue = pd.read_csv(PIPELINE_OUTPUT / "refresh_queue.csv")
model_results = json.loads((PIPELINE_OUTPUT / "model_results.json").read_text(encoding="utf-8"))
summary = json.loads((PIPELINE_OUTPUT / "summary.json").read_text(encoding="utf-8"))

assert queue["final_rank"].is_monotonic_increasing
assert queue["content_id"].is_unique
assert {"suggested_action", "final_reason_codes", "confidence", "final_refresh_score"}.issubset(queue.columns)

print(f"Queue rows: {len(queue):,}")
print(f"Best model: {model_results['best_model']['name']}")
print(f"Observed declining-label rate: {model_results['target_positive_rate']:.3f}")
print("Top 10 ranked actions:")
display(queue[[
    "final_rank", "content_id", "final_refresh_score", "confidence",
    "suggested_action", "final_reason_codes", "impressions_90d", "sessions_90d",
]].head(10))

Prepared 30,000 rows from 30,000 raw rows
Wrote /workspaces/flyrank-ml-internship/data/processed/refresh_feature_vector.csv


Wrote baseline queue: /workspaces/flyrank-ml-internship/data/processed/baseline_refresh_queue.csv
Top-50 declining rate (full data, not the evaluated holdout Precision@50): 0.340


Trained 3 models on 30,000 rows
Split strategy: client_holdout
Best model: random_forest
Wrote predictions: /workspaces/flyrank-ml-internship/data/processed/model_predictions.csv
Wrote model results: /workspaces/flyrank-ml-internship/outputs/model_results.json


Wrote final refresh queue: /workspaces/flyrank-ml-internship/outputs/refresh_queue.csv
Wrote model report: /workspaces/flyrank-ml-internship/outputs/model_report.md
Wrote charts in: /workspaces/flyrank-ml-internship/outputs/charts
Queue rows: 30,000
Best model: random_forest
Observed declining-label rate: 0.542
Top 10 ranked actions:


,final_rank,content_id,final_refresh_score,confidence,suggested_action,final_reason_codes,impressions_90d,sessions_90d
0,1,content_1f080331fa2b,81.928467,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,12834,66
1,2,content_6aa43079fb0c,81.728449,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,8064,23
2,3,content_d6570c51c9bd,81.639118,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,2498,9
3,4,content_e04eb9549989,80.804986,medium,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,3393,5
4,5,content_72e800a9c214,80.801530,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,13790,27
5,6,content_9b6df29f7889,80.752578,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,1622,20
6,7,content_b69288c5e701,80.632372,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,5811,14
7,8,content_ba6f9dfcbca1,80.439403,medium,refresh,declining_with_demand|model_decline_risk|visib...,4366,8
8,9,content_b1d593faf9c6,80.204325,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|low...,2655,92
9,10,content_bb6ebb5ec8c8,80.146808,high,refresh_and_review_ctr,declining_with_demand|low_ctr_visible_page|mod...,2621,10


## 2. Intended use and limits

**Intended use:** an editor or SEO analyst uses the ranked queue to decide which pseudonymous content items to inspect first. The queue is a decision-support artifact for a review meeting, not a production publishing system.

**Archetype to action mapping:**

| Archetype | Signal pattern | Suggested action | Review question |
|---|---|---|---|
| Visible decline | demand plus a down trend or elevated model risk | `refresh` | Is the page still aligned with the search intent and current facts? |
| Visible but low CTR | impressions and visible position with low CTR | `refresh_and_review_ctr` | Does the title/snippet accurately communicate the page's value? |
| Engaged but weak experience | sessions with low engagement or scroll signal | `refresh_and_review_engagement` | Is the page useful, readable, and easy to navigate? |
| Thin visible page | visibility with a thin content signal | `expand_and_refresh` | What evidence-backed coverage is missing? |
| Unclear or low-signal item | weak evidence or lower score | `monitor` | What new observation would justify escalation? |

**Limits:** this is one anonymized snapshot, the label is a measured trend proxy, and the grouped validation supports directional prioritization rather than a guarantee for a new client or future period. A refresh is not observed as a treatment here, so the queue cannot claim that acting on it will improve traffic, rankings, or revenue.

In [2]:
def has_reason(reason_text: str, reason: str) -> bool:
    return reason in set(str(reason_text).split("|"))


def assign_archetype(row: pd.Series) -> str:
    reasons = str(row["final_reason_codes"])
    if has_reason(reasons, "thin_visible_page"):
        return "thin_visible_page"
    if has_reason(reasons, "ctr_review_candidate"):
        return "visible_low_ctr"
    if has_reason(reasons, "engagement_review_candidate"):
        return "engaged_weak_experience"
    if has_reason(reasons, "declining_with_demand") or has_reason(reasons, "model_decline_risk"):
        return "visible_decline"
    return "unclear_low_signal"


queue["archetype"] = queue.apply(assign_archetype, axis=1)
queue["review_effort"] = np.select(
    [queue["confidence"].eq("high"), queue["confidence"].eq("medium")],
    ["focused_review", "sampled_review"],
    default="monitor_batch",
)
# This is a transparent triage proxy, not expected revenue or causal value.
queue["review_value_proxy"] = (
    queue["impressions_90d"] * queue["best_model_probability"]
).round(2)

archetype_summary = (
    queue.groupby("archetype", observed=False)
    .agg(
        rows=("content_id", "size"),
        median_score=("final_refresh_score", "median"),
        median_impressions=("impressions_90d", "median"),
        median_review_value_proxy=("review_value_proxy", "median"),
    )
    .sort_values("median_score", ascending=False)
)
print("Archetype to action summary:")
display(archetype_summary.round(2))

print("Action mix:")
display(queue["suggested_action"].value_counts().rename_axis("action").to_frame("rows"))

trend_summary = (
    queue.groupby("trend_direction", observed=False)
    .agg(
        rows=("content_id", "size"),
        median_score=("final_refresh_score", "median"),
        refresh_or_review_rate=("suggested_action", lambda values: values.ne("monitor").mean()),
    )
    .sort_values("rows", ascending=False)
)
print("Observed decay/refresh pattern in this snapshot:")
display(trend_summary.round(3))
print("Cost/value framing: prioritize high-score items with visible demand for focused human review; use the proxy only to compare triage order, never as money or uplift.")

Archetype to action summary:


,rows,median_score,median_impressions,median_review_value_proxy
archetype,,,,
visible_low_ctr,9741,60.52,3022.0,1777.13
visible_decline,6295,58.36,359.0,225.69
engaged_weak_experience,3415,57.22,7387.0,3700.23
thin_visible_page,82,50.57,708.5,333.50
unclear_low_signal,10467,35.58,25.0,12.50


Action mix:


,rows
action,
monitor,13069
refresh,8207
refresh_and_review_ctr,6655
refresh_and_review_engagement,1987
expand_and_refresh,82


Observed decay/refresh pattern in this snapshot:


,rows,median_score,refresh_or_review_rate
trend_direction,,,
down,16262,58.693,0.838
stable,5962,52.099,0.343
up,4388,47.745,0.285
new,2236,6.996,0.003
flat,1152,24.357,0.006


Cost/value framing: prioritize high-score items with visible demand for focused human review; use the proxy only to compare triage order, never as money or uplift.


## 3. Human review + the no-go list

Every queue row needs a human decision before any content change. The reviewer should verify the page, current search intent, factual freshness, brand or legal constraints, and whether the observed signal is large enough to matter. The reviewer records **accept**, **defer**, or **reject** with a short reason.

**Do not automate:** publishing or deleting content; changing claims in regulated or safety-sensitive topics; changing canonical URLs, redirects, internal-link architecture, or metadata at scale; inferring user intent from private queries; treating a model score as a quality verdict; or claiming that a refresh caused a recovery. The model has no authority to make those decisions.

In [3]:
review_rules = pd.DataFrame([
    {
        "stage": "identity_and_scope",
        "human_check": "Confirm the pseudonymous item is in scope and the queue row is current.",
        "decision": "accept, defer, or reject",
    },
    {
        "stage": "evidence",
        "human_check": "Inspect the observed demand, position, CTR, engagement, age, and reason codes together.",
        "decision": "record the evidence used",
    },
    {
        "stage": "content_review",
        "human_check": "Check search intent, factual accuracy, usefulness, accessibility, and editorial context.",
        "decision": "choose an action or defer",
    },
    {
        "stage": "approval",
        "human_check": "Require a human owner for any rewrite, expansion, redirect, or publication change.",
        "decision": "human approval required",
    },
])
no_go_list = [
    "automatic publishing, deletion, or rewriting",
    "automatic changes to URLs, canonicals, redirects, or internal links",
    "automated decisions in regulated or safety-sensitive content",
    "using client_id or content_id as a quality signal",
    "treating model probability as a causal forecast or revenue estimate",
    "claiming a refresh caused a recovery without an appropriate design",
]
print("Human review rules:")
display(review_rules)
print("No-go list:")
for item in no_go_list:
    print(f"- {item}")

assert len(review_rules) == 4
assert len(no_go_list) == 6

Human review rules:


,stage,human_check,decision
0,identity_and_scope,Confirm the pseudonymous item is in scope and ...,"accept, defer, or reject"
1,evidence,"Inspect the observed demand, position, CTR, en...",record the evidence used
2,content_review,"Check search intent, factual accuracy, usefuln...",choose an action or defer
3,approval,"Require a human owner for any rewrite, expansi...",human approval required


No-go list:
- automatic publishing, deletion, or rewriting
- automatic changes to URLs, canonicals, redirects, or internal links
- automated decisions in regulated or safety-sensitive content
- using client_id or content_id as a quality signal
- treating model probability as a causal forecast or revenue estimate
- claiming a refresh caused a recovery without an appropriate design


## 4. Monitoring / retrain triggers

This is a light monitoring contract for the next snapshot, not a production alerting system. Recompute the queue and compare like with like: same label definition, same grouped validation design, and the same base-rate reference.

Retrain or pause the playbook for review when any of these occurs:

- the feature row count or required-field coverage changes materially, such as a 20% row-count shift or a newly missing required field;
- the declining-label rate moves by more than 10 percentage points from the recorded snapshot;
- grouped average precision or Precision@50 falls below the transparent baseline, or falls by roughly 10% relative to this receipt;
- the action mix changes sharply without a documented data or business change;
- a feature definition, measurement window, client mix, or label rule changes;
- sampled human reviews show repeated false positives, false negatives, or stale reason codes.

A future refresh outcome can be monitored as a directional check, but this notebook does not estimate causal uplift.

In [4]:
best_name = model_results["best_model"]["name"]
best_metrics = model_results["models"][best_name]
baseline_metrics = model_results["baseline"]

monitoring_receipt = {
    "rows_scored": int(len(queue)),
    "target_positive_rate": float(model_results["target_positive_rate"]),
    "best_model": best_name,
    "grouping_for_recorded_week5_metrics": model_results["split_strategy"],
    "model_average_precision_reference": float(best_metrics["average_precision"]),
    "model_precision_at_50_reference": float(best_metrics["precision_at_50"]),
    "baseline_average_precision_reference": float(baseline_metrics["baseline_average_precision"]),
    "baseline_precision_at_50_reference": float(baseline_metrics["baseline_precision_at_50"]),
    "trigger_thresholds": {
        "row_count_relative_change": 0.20,
        "target_rate_absolute_change": 0.10,
        "metric_relative_drop": 0.10,
    },
}
monitoring_table = pd.DataFrame([
    ["Data contract", "row count shifts >= 20% or required fields become missing", "pause and inspect inputs"],
    ["Label mix", "declining-label rate shifts >= 10 percentage points", "recheck label and windows"],
    ["Ranking quality", "grouped AP or Precision@50 below baseline or down >= 10%", "revalidate and retrain"],
    ["Action mix", "large unexplained change in action proportions", "inspect drift and reason codes"],
    ["Human review", "repeated false positives, false negatives, or stale reasons", "revise rules/features"],
], columns=["monitor", "trigger", "response"])
print("Monitoring receipt:")
display(monitoring_table)
print("Recorded metric references:")
display(pd.DataFrame([monitoring_receipt]).T.rename(columns={0: "value"}))

Monitoring receipt:


,monitor,trigger,response
0,Data contract,row count shifts >= 20% or required fields bec...,pause and inspect inputs
1,Label mix,declining-label rate shifts >= 10 percentage p...,recheck label and windows
2,Ranking quality,grouped AP or Precision@50 below baseline or d...,revalidate and retrain
3,Action mix,large unexplained change in action proportions,inspect drift and reason codes
4,Human review,"repeated false positives, false negatives, or ...",revise rules/features


Recorded metric references:


,value
rows_scored,30000
target_positive_rate,0.542067
best_model,random_forest
grouping_for_recorded_week5_metrics,client_holdout
model_average_precision_reference,0.610058
model_precision_at_50_reference,0.68
baseline_average_precision_reference,0.467607
baseline_precision_at_50_reference,0.24
trigger_thresholds,"{'row_count_relative_change': 0.2, 'target_rat..."


## 5. Exports for the paper

The notebook regenerates the full ranked queue at `work/outputs/refresh_queue.csv`; it is intentionally ignored because it contains row-level data. The paper can cite the committed metrics receipt and figures without committing the queue itself.

Committed receipts and reusable figures:

- `work/outputs/action_playbook_metrics.json`
- `work/outputs/model_results.json`
- `work/outputs/action_playbook_summary.json`
- `work/figures/action_mix.svg`
- `work/figures/archetype_mix.svg`

The queue includes rank, score, confidence, suggested action, reason codes, and review context. It is an input to human review, not a production action feed.

In [5]:
from ml_utils import simple_svg_bar_chart

queue_export_path = WORK_OUTPUT / "refresh_queue.csv"
metrics_export_path = WORK_OUTPUT / "action_playbook_metrics.json"
model_export_path = WORK_OUTPUT / "model_results.json"
summary_export_path = WORK_OUTPUT / "action_playbook_summary.json"

queue.to_csv(queue_export_path, index=False)
shutil.copyfile(PIPELINE_OUTPUT / "model_results.json", model_export_path)

metrics_payload = {
    "rows_scored": int(len(queue)),
    "target_positive_rate": float(model_results["target_positive_rate"]),
    "best_model": best_name,
    "model_average_precision": float(best_metrics["average_precision"]),
    "model_precision_at_50": float(best_metrics["precision_at_50"]),
    "baseline_average_precision": float(baseline_metrics["baseline_average_precision"]),
    "baseline_precision_at_50": float(baseline_metrics["baseline_precision_at_50"]),
    "action_counts": {str(key): int(value) for key, value in queue["suggested_action"].value_counts().items()},
    "confidence_counts": {str(key): int(value) for key, value in queue["confidence"].value_counts().items()},
    "archetype_counts": {str(key): int(value) for key, value in queue["archetype"].value_counts().items()},
    "intended_use": "human-reviewed prioritization of pseudonymous content items",
    "causal_claim": False,
    "queue_output": "work/outputs/refresh_queue.csv",
    "figures": ["work/figures/action_mix.svg", "work/figures/archetype_mix.svg"],
}
metrics_export_path.write_text(json.dumps(metrics_payload, indent=2, sort_keys=True), encoding="utf-8")
summary_export_path.write_text(json.dumps({
    "rows_scored": int(len(queue)),
    "high_confidence_rows": int((queue["confidence"] == "high").sum()),
    "top_queue_score": float(queue["final_refresh_score"].max()),
    "top_ranked_action": str(queue.iloc[0]["suggested_action"]),
    "review_effort_counts": {str(key): int(value) for key, value in queue["review_effort"].value_counts().items()},
}, indent=2, sort_keys=True), encoding="utf-8")

simple_svg_bar_chart(
    "Action mix",
    queue["suggested_action"].value_counts().index.tolist(),
    queue["suggested_action"].value_counts().values.tolist(),
    WORK_FIGURES / "action_mix.svg",
    color="#426B69",
)
simple_svg_bar_chart(
    "Archetype mix",
    queue["archetype"].value_counts().index.tolist(),
    queue["archetype"].value_counts().values.tolist(),
    WORK_FIGURES / "archetype_mix.svg",
    color="#4E79A7",
)

assert queue_export_path.exists()
assert metrics_export_path.exists() and model_export_path.exists() and summary_export_path.exists()
assert (WORK_FIGURES / "action_mix.svg").exists()
assert (WORK_FIGURES / "archetype_mix.svg").exists()
print(f"Exported queue: {queue_export_path}")
print(f"Exported metrics: {metrics_export_path}")
print(f"Exported figures: {WORK_FIGURES}")

Exported queue: /workspaces/flyrank-ml-internship/work/outputs/refresh_queue.csv
Exported metrics: /workspaces/flyrank-ml-internship/work/outputs/action_playbook_metrics.json
Exported figures: /workspaces/flyrank-ml-internship/work/figures


## Self-check

Before submitting, I confirm:

- [x] Ranked actions and inspectable reason codes are generated from the real queue.
- [x] Archetypes map to practical actions, including the observed decay/refresh pattern.
- [x] Intended use, limits, review rules, cost/value framing, and monitoring triggers are stated.
- [x] No-go cases explicitly keep publishing, deletion, sensitive decisions, and causal claims under human control.
- [x] The queue is regenerated under `work/outputs/`, while metrics receipts and reusable figures are committed.
- [x] The notebook is designed to run top to bottom with public-safe pseudonymous identifiers only.
- [x] Claims use observed, measured, directional, and decision-support language.